In [1]:
from pyspark import SparkContext

sc = SparkContext("local", "Simple App")

filename = "data/A.txt"
file_A = sc.textFile(filename).cache()

filename = "data/B.txt"
file_B = sc.textFile(filename).cache()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/27 10:23:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read Vectors and Matrices

In [61]:
filename = "data/M.txt"
file_matrix = sc.textFile(filename).cache()

filename = "data/V.txt"
file_V = sc.textFile(filename).cache()
filename = "data/W.txt"
file_W = sc.textFile(filename).cache()


def format_vector(x):
    x = x.split(' ')
    x_0 = int(x[0])
    x_1 = float(x[1])
    return (x_0, x_1)

V = file_V.map(format_vector)
V.collect()
W = file_W.map(format_vector)
W.collect()


def format_matrix(x):
    x = x.split(' ')
    x_0 = int(x[0])
    x_1 = int(x[1])
    x_2 = float(x[2])
    return (x_0, x_1, x_2)

M = file_matrix.map(format_matrix)
M.collect()

[(1, 1, 3.2),
 (1, 2, 2.4),
 (1, 3, 7.0),
 (1, 4, 2.0),
 (2, 2, 7.1),
 (2, 3, -1.0),
 (3, 3, 1.0)]

In [62]:
print(V.collect())
print(W.collect())

[(1, 1.5), (2, 5.0), (4, 1.3), (7, 3.0)]
[(1, -1.5), (2, 2.0), (3, 2.3), (4, 2.0), (6, 2.5)]


In [63]:
V.join(W).map(lambda x: (x[1][0]*x[1][1])).sum()

10.35

In [77]:
def f1(x):
    return ((x[1][0][0], x[0]), round(x[1][0][1]*x[1][1], 2))

join_by_j = M.map(lambda x: (x[1],(x[0], x[2]))).join(V)
join_by_j = join_by_j.map(f1).sortByKey()
join_by_j.collect()

[((1, 1), 4.8), ((1, 2), 12.0), ((1, 4), 2.6), ((2, 2), 35.5)]

In [79]:
join_by_j.map(lambda x: (x[0][0], x[1])).reduceByKey(lambda a,b: round(a+b, 2)).sortByKey().collect()

[(1, 19.4), (2, 35.5)]